In [1]:
from pathlib import Path

import json
import math
import random
import gc

import numpy as np
import pandas as pd

import torch

from tqdm.auto import tqdm


from datasets import Dataset


from transformers import (
    M2M100Tokenizer,
    M2M100ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup,
)


import sacrebleu

In [2]:
SEED = 42


random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)


if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [17]:
PROJECT_ROOT = Path(
    r"D:\dev\projects\fourlang_translation"
)


DATA_DIR = (
    PROJECT_ROOT
    /
    "data"
    /
    "clean"
    /
    "en_uz"
    /
    "exp2"
)


OUTPUT_DIR = (
    PROJECT_ROOT
    /
    "models"
    /
    "m2m100_ft"
    /
    "exp2_en_uz_42384"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(DATA_DIR)
print(OUTPUT_DIR)

D:\dev\projects\fourlang_translation\data\clean\en_uz\exp2
D:\dev\projects\fourlang_translation\models\m2m100_ft\exp2_en_uz_42384


In [18]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(device)


if torch.cuda.is_available():

    print(
        torch.cuda.get_device_name(0)
    )

cuda
NVIDIA GeForce RTX 5060 Laptop GPU


In [19]:
train_df = pd.read_json(
    DATA_DIR /
    "train_50k.jsonl",
    lines=True
)


valid_df = pd.read_json(
    DATA_DIR /
    "validation.jsonl",
    lines=True
)


test_df = pd.read_json(
    DATA_DIR /
    "test.jsonl",
    lines=True
)


print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

(90000, 4)
(5000, 4)
(5000, 4)


In [20]:
print(
    train_df[
        [
            "src_lang",
            "tgt_lang"
        ]
    ]
    .head()
)


print(
    train_df["src_lang"].unique()
)


print(
    train_df["tgt_lang"].unique()
)

  src_lang tgt_lang
0       uz       en
1       uz       en
2       en       uz
3       uz       en
4       uz       en
<ArrowStringArray>
['uz', 'en']
Length: 2, dtype: str
<ArrowStringArray>
['en', 'uz']
Length: 2, dtype: str


In [21]:
train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)


valid_dataset = Dataset.from_pandas(
    valid_df,
    preserve_index=False
)


print(train_dataset)

Dataset({
    features: ['src_lang', 'tgt_lang', 'src_text', 'tgt_text'],
    num_rows: 90000
})


In [22]:
MODEL_NAME = (
    "facebook/m2m100_418M"
)


tokenizer = M2M100Tokenizer.from_pretrained(
    MODEL_NAME,
    src_lang="en",
    tgt_lang="uz"
)


model = M2M100ForConditionalGeneration.from_pretrained(
    MODEL_NAME
)


model.to(device)


print("model loaded")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model loaded


In [23]:
print(
    tokenizer.lang_code_to_token["en"]
)


print(
    tokenizer.lang_code_to_token["uz"]
)

__en__
__uz__


In [24]:
MAX_LENGTH = 128


def preprocess(example):


    src_lang = example["src_lang"]

    tgt_lang = example["tgt_lang"]


    assert src_lang in [
        "en",
        "uz"
    ], example


    assert tgt_lang in [
        "en",
        "uz"
    ], example



    tokenizer.src_lang = src_lang


    tokenizer.tgt_lang = tgt_lang



    encoded = tokenizer(
        example["src_text"],

        text_target=
        example["tgt_text"],

        max_length=MAX_LENGTH,

        truncation=True
    )


    return {

        "input_ids":
            encoded["input_ids"],


        "attention_mask":
            encoded["attention_mask"],


        "labels":
            encoded["labels"]

    }

In [25]:
token_train = train_dataset.map(
    preprocess,
    remove_columns=
    train_dataset.column_names
)


token_valid = valid_dataset.map(
    preprocess,
    remove_columns=
    valid_dataset.column_names
)


print(token_train)

Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 90000
})


In [27]:
from torch.utils.data import DataLoader

collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)



train_loader = DataLoader(
    token_train,
    batch_size=2,
    shuffle=True,
    collate_fn=collator
)



valid_loader = DataLoader(
    token_valid,
    batch_size=4,
    shuffle=False,
    collate_fn=collator
)

In [29]:
EPOCHS = 3


LR = 2e-5


GRAD_ACCUMULATION = 8

In [30]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
)

In [31]:
total_steps = (
    len(train_loader)
    //
    GRAD_ACCUMULATION
    *
    EPOCHS
)


scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=
    int(total_steps*0.05),
    num_training_steps=
    total_steps
)

In [32]:
@torch.no_grad()
def evaluate():

    model.eval()

    losses=[]


    for batch in valid_loader:


        batch={
            k:v.to(device)
            for k,v in batch.items()
        }


        outputs=model(
            **batch
        )


        losses.append(
            outputs.loss.item()
        )


    return np.mean(losses)

In [33]:
history=[]


for epoch in range(EPOCHS):


    model.train()


    total_loss=0


    optimizer.zero_grad()


    loop=tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}"
    )


    for step,batch in enumerate(loop):


        batch={
            k:v.to(device)
            for k,v in batch.items()
        }


        outputs=model(
            **batch
        )


        loss = (
            outputs.loss
            /
            GRAD_ACCUMULATION
        )


        loss.backward()



        if (
            step+1
        ) % GRAD_ACCUMULATION == 0:


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )


            optimizer.step()

            scheduler.step()

            optimizer.zero_grad()



        total_loss += (
            outputs.loss.item()
        )


        loop.set_postfix(
            loss=
            outputs.loss.item()
        )



    valid_loss=evaluate()



    result={

        "epoch":
            epoch+1,

        "train_loss":
            total_loss/
            len(train_loader),

        "valid_loss":
            valid_loss

    }



    history.append(result)


    print(result)

Epoch 1:   0%|          | 0/45000 [00:00<?, ?it/s]

KeyboardInterrupt: 